# Fermionic 3D toric code, L=2 OBC — $(h_x, h_z)$ plane

Four sign architectures (`asymm` no-head control, `anaC_k0` frozen analytic
head, `pt2sf`/`pt2sfc` perturbative-sign heads) graded against dense ED on
the full $2^{12}=4096$-dim Hilbert space, across the $4\times4$ grid
$h_x, h_z \in \{0.0, 0.2, 0.5, 1.0\}$ (16 field points, 64 arm-points).

Mirrors the 2D peer project's phase-4b/4c plane figures
(`2D-TC/analysis/05_phase4b_plane.ipynb`, branch `doubled-semion`): a P1-style
1x4 arms plane (shared `LogNorm`, `Blues`, annotated cells) for the achieved
relative energy error and trained infidelity, and a P6-style two-row
achieved-vs-gate-0-ceiling comparison (`Blues` over `Purples`, exact-zero
cells masked gray and labeled *exact*) using the sign-fidelity ceilings from
`analysis/scripts/sign_fidelity_ftc.py --out_tag plane`.

**Data status:** the plane campaign (`results/fermionic_plane_L2/`) is being
launched; until it lands, this notebook runs on the $h_z=0$ stand-in ladder
at `results/fermionic_hx_ladder/` (§1 prints a warning when this fallback is
active) so every cell — including the NaN/missing-point handling — is
exercised end-to-end. Re-run unchanged once the real 16-point campaign data
arrives.

## 1. Config + loaders

In [ ]:
# %% 1. CONFIG — knobs live here -------------------------------------------
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})

HX_VALUES = [0.0, 0.2, 0.5, 1.0]
HZ_VALUES = [0.0, 0.2, 0.5, 1.0]
ARMS = ["asymm", "anaC_k0", "pt2sf", "pt2sfc"]
ARM_LABEL = {
    "asymm": "asymm (no head)",
    "anaC_k0": r"anaC, $\kappa$=0",
    "pt2sf": "pt2 head (sf)",
    "pt2sfc": "pt2 head (sfc)",
}
# Okabe-Ito palette (plot-style-spec)
ARM_COLOR = {"asymm": "#0072B2", "anaC_k0": "#E69F00", "pt2sf": "#009E73", "pt2sfc": "#CC79A7"}
# arm -> matching gate-0 head name in the sign_fidelity_ftc.py F_s / one_minus_F_s dicts
ARM_HEAD = {"asymm": "plus", "anaC_k0": "anaC", "pt2sf": "pt2", "pt2sfc": "pt2"}

INTERP = False  # 4x4 grid is sparse -> annotated discrete imshow cells by default;
                # set True for the peer notebook's griddata-cubic continuous field style

ROOT = Path("../../results/fermionic_plane_L2")
FALLBACK_ROOT = Path("../../results/fermionic_hx_ladder")
GATE0_PATH = Path("../../results/fermionic_gate0/2x2x2_OBC_plane_gate0.json")
FIGS = Path("../figs")

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from analysis.scripts.plane_summary import build as build_plane_summary

if ROOT.is_dir() and any(ROOT.glob("gridinv_fermionic_*.json")):
    SUMMARY = build_plane_summary(ROOT)
    DATA_SRC = str(ROOT)
else:
    warnings.warn(
        f"{ROOT} not found or empty -- falling back to the hz=0 stand-in ladder at "
        f"{FALLBACK_ROOT} (only hz=0 cells can be populated; everything else stays "
        "NaN). Re-run this notebook unchanged once the real plane campaign lands.")
    SUMMARY = build_plane_summary(FALLBACK_ROOT)
    DATA_SRC = f"{FALLBACK_ROOT} (STAND-IN)"

SUMMARY_BY_KEY = {(r["hx"], r["hz"], r["arm"]): r for r in SUMMARY}
print(f"data source: {DATA_SRC}")
print(f"loaded {len(SUMMARY)} rows; grid {len(HX_VALUES)}x{len(HZ_VALUES)} x {len(ARMS)} arms "
      f"= {len(HX_VALUES) * len(HZ_VALUES) * len(ARMS)} cells (missing -> NaN)")


## 2. Headline table

In [ ]:
# %% 2. headline table (16 points x 4 arms: rel, 1-fidelity, diverged) ------
hdr = (f"{'hx':>5} {'hz':>5} {'arm':<10} {'rel_err':>10} "
       f"{'1-fidelity':>11} {'diverged':>9} {'last_step':>9}")
print(hdr)
print("-" * len(hdr))
for hz in HZ_VALUES:
    for hx in HX_VALUES:
        for arm in ARMS:
            r = SUMMARY_BY_KEY.get((hx, hz, arm))
            if r is None:
                print(f"{hx:>5} {hz:>5} {arm:<10} {'--':>10} {'--':>11} {'--':>9} {'--':>9}")
                continue
            rel_s = f"{r['rel']:.2e}" if r.get("rel") is not None else "--"
            fid_s = f"{1 - r['fidelity']:.2e}" if r.get("fidelity") is not None else "--"
            print(f"{hx:>5} {hz:>5} {arm:<10} {rel_s:>10} {fid_s:>11} "
                  f"{str(r['diverged']):>9} {str(r['last_step']):>9}")
    print()


## 3. Arms plane — relative energy error (P1-style)

1x4 panels, one per arm, shared `LogNorm` over `Blues`, cells annotated with
the achieved value. Missing (hx, hz, arm) points render as blank gray cells
(`np.nan`, masked).

In [ ]:
# %% 3. shared plotting helpers ----------------------------------------------
def build_grid(field_fn):
    """(len(HZ_VALUES), len(HX_VALUES)) matrix; field_fn(hx, hz) -> float | nan."""
    W = np.full((len(HZ_VALUES), len(HX_VALUES)), np.nan)
    for i, hz in enumerate(HZ_VALUES):
        for j, hx in enumerate(HX_VALUES):
            W[i, j] = field_fn(hx, hz)
    return W


def rel_err_field(hx, hz, arm):
    r = SUMMARY_BY_KEY.get((hx, hz, arm))
    return np.nan if r is None or r.get("rel") is None else r["rel"]


def infid_field(hx, hz, arm):
    r = SUMMARY_BY_KEY.get((hx, hz, arm))
    return np.nan if r is None or r.get("fidelity") is None else 1.0 - r["fidelity"]


def _positive_finite(mats):
    parts = [m[np.isfinite(m) & (m > 0)] for m in mats.values()]
    parts = [p for p in parts if p.size]
    return np.concatenate(parts) if parts else np.array([])


def plot_arms_plane(field_fn, title, cbar_label, cmap="Blues", zero_as_exact=False,
                     savefig_name=None):
    """P1-style 1x4 arms panel. field_fn(hx, hz, arm) -> float (nan if missing)."""
    mats = {arm: build_grid(lambda hx, hz, a=arm: field_fn(hx, hz, a)) for arm in ARMS}
    pos = _positive_finite(mats)
    if pos.size == 0:
        print(f"[skip] {title}: no finite positive data yet")
        return
    vmin, vmax = pos.min(), pos.max()
    norm = LogNorm(vmin=vmin, vmax=max(vmax, vmin * (1 + 1e-9)))

    fig, axes = plt.subplots(1, len(ARMS), figsize=(3.5 * len(ARMS) + 1.3, 4.3),
                              sharey=True, constrained_layout=True)

    if INTERP:
        from scipy.interpolate import griddata
        g = np.linspace(min(HX_VALUES), max(HX_VALUES), 161)
        gz = np.linspace(min(HZ_VALUES), max(HZ_VALUES), 161)
        GX, GY = np.meshgrid(g, gz)

    for ax, arm in zip(axes, ARMS):
        ax.grid(False)
        M = mats[arm]
        if INTERP:
            pts, logv = [], []
            for i, hz in enumerate(HZ_VALUES):
                for j, hx in enumerate(HX_VALUES):
                    if np.isfinite(M[i, j]) and M[i, j] > 0:
                        pts.append((hx, hz)); logv.append(np.log10(M[i, j]))
            if len(pts) >= 4:
                F = np.clip(griddata(np.array(pts), np.array(logv), (GX, GY), method="cubic"),
                            np.log10(vmin), np.log10(vmax))
                im = ax.imshow(10 ** np.ma.masked_invalid(F), origin="lower",
                               extent=(min(HX_VALUES), max(HX_VALUES),
                                       min(HZ_VALUES), max(HZ_VALUES)),
                               cmap=cmap, norm=norm, interpolation="bilinear")
            else:
                im = ax.imshow(np.ma.masked_all((2, 2)), origin="lower", cmap=cmap, norm=norm)
            ax.set_facecolor("0.88")
            for (hx, hz), lv in zip(pts, logv):
                dark = norm(10 ** lv) > 0.6
                ax.plot(hx, hz, ".", ms=3, color="white" if dark else "#1a1a1a", zorder=4)
                ax.text(hx, hz, f"{10 ** lv:.1e}", ha="center", va="center", fontsize=7,
                        color="white" if dark else "#1a1a1a", zorder=5)
            ax.set_xticks(HX_VALUES); ax.set_yticks(HZ_VALUES)
        else:
            Mm = np.ma.masked_invalid(M)
            im = ax.imshow(Mm, origin="lower", cmap=cmap, norm=norm, interpolation="nearest")
            ax.set_facecolor("0.85")
            for i, hz in enumerate(HZ_VALUES):
                for j, hx in enumerate(HX_VALUES):
                    v = M[i, j]
                    if not np.isfinite(v):
                        continue
                    if zero_as_exact and v <= 0:
                        ax.text(j, i, "exact", ha="center", va="center", fontsize=7.5,
                                color="#1a1a1a", style="italic")
                    else:
                        dark = norm(max(v, vmin)) > 0.6
                        ax.text(j, i, f"{v:.1e}", ha="center", va="center", fontsize=8,
                                color="white" if dark else "#1a1a1a")
            ax.set_xticks(range(len(HX_VALUES))); ax.set_xticklabels(HX_VALUES)
            ax.set_yticks(range(len(HZ_VALUES))); ax.set_yticklabels(HZ_VALUES)
        ax.set(xlabel="$h_x$", title=ARM_LABEL[arm])
    axes[0].set(ylabel="$h_z$")
    cb = fig.colorbar(im, ax=axes, shrink=0.85, pad=0.02)
    cb.set_label(cbar_label)
    fig.suptitle(title, y=1.05)
    if savefig_name:
        pass
    # plt.savefig(FIGS / f"fermionic_plane_L2_{savefig_name}.png", dpi=300, bbox_inches="tight")
    plt.show()


plot_arms_plane(rel_err_field, "Fermionic L=2 OBC plane: relative energy error per arm",
                r"rel. energy error $|E-E_0|/|E_0|$", cmap="Blues", savefig_name="relerr")


## 4. Arms plane — trained infidelity $1-F$

In [ ]:
# %% 4. same layout for trained infidelity -----------------------------------
plot_arms_plane(infid_field, "Fermionic L=2 OBC plane: trained infidelity per arm",
                r"trained infidelity $1-F$", cmap="Purples", savefig_name="infid")


## 5. Achieved vs gate-0 ceiling (P6-style)

Top row: achieved trained infidelity $1-F$ (same data as §4, `Blues`). Bottom
row: the exact gate-0 sign-fidelity ceiling $1-F_s$ for the head each arm
uses (`asymm`→`plus`, `anaC_k0`→`anaC`, `pt2sf`/`pt2sfc`→`pt2`), `Purples`,
from `results/fermionic_gate0/2x2x2_OBC_plane_gate0.json`
(`sign_fidelity_ftc.py --out_tag plane`). Exactly-zero ceiling cells are
masked gray and labeled *exact*, matching the peer notebook's convention.

In [ ]:
# %% 5. in-vivo achieved infidelity vs exact gate-0 ceiling -------------------
def load_gate0(path):
    if not path.exists():
        warnings.warn(f"{path} not found -- run "
                       "`analysis/scripts/sign_fidelity_ftc.py ... --out_tag plane` "
                       "first; section 5 will be skipped.")
        return None
    d = json.load(open(path))
    return {(round(p["hx"], 6), round(p["hz"], 6)): p for p in d["points"]}


GATE0 = load_gate0(GATE0_PATH)


def ceiling_field(hx, hz, arm):
    if GATE0 is None:
        return np.nan
    p = GATE0.get((round(hx, 6), round(hz, 6)))
    if p is None:
        return np.nan
    return p["one_minus_F_s"].get(ARM_HEAD[arm], np.nan)


def plot_invivo_vs_ceiling():
    if GATE0 is None:
        print("[skip] section 5: no gate-0 plane ceiling file yet")
        return
    W_ach = {arm: build_grid(lambda hx, hz, a=arm: infid_field(hx, hz, a)) for arm in ARMS}
    W_ceil = {arm: build_grid(lambda hx, hz, a=arm: ceiling_field(hx, hz, a)) for arm in ARMS}

    pos_a, pos_c = _positive_finite(W_ach), _positive_finite(W_ceil)
    norm_a = LogNorm(vmin=pos_a.min(), vmax=pos_a.max()) if pos_a.size else None
    norm_c = LogNorm(vmin=pos_c.min(), vmax=pos_c.max()) if pos_c.size else None

    fig, axes = plt.subplots(2, len(ARMS), figsize=(3.5 * len(ARMS) + 1.3, 8.2),
                              sharex=True, sharey=True, constrained_layout=True)
    rows = [(W_ach, norm_a, "Blues", "achieved\n1-F"),
            (W_ceil, norm_c, "Purples", "gate-0 ceiling\n$1-F_s$")]
    for r, (W, norm_r, cmap_r, row_lab) in enumerate(rows):
        for ax, arm in zip(axes[r], ARMS):
            ax.grid(False)
            M = W[arm]
            Mm = np.ma.masked_invalid(M)
            Mm = np.ma.masked_less_equal(Mm, 0.0)
            im = (ax.imshow(Mm, origin="lower", cmap=cmap_r, norm=norm_r, interpolation="nearest")
                  if norm_r is not None else
                  ax.imshow(Mm, origin="lower", cmap=cmap_r, interpolation="nearest"))
            ax.set_facecolor("0.9")
            for i, hz in enumerate(HZ_VALUES):
                for j, hx in enumerate(HX_VALUES):
                    v = M[i, j]
                    if not np.isfinite(v):
                        continue
                    if v <= 0:
                        ax.text(j, i, "exact", ha="center", va="center", fontsize=7.5,
                                color="#1a1a1a", style="italic")
                    else:
                        dark = (norm_r(v) > 0.6) if norm_r is not None else False
                        ax.text(j, i, f"{v:.1e}", ha="center", va="center", fontsize=8,
                                color="white" if dark else "#1a1a1a")
            ax.set_xticks(range(len(HX_VALUES))); ax.set_xticklabels(HX_VALUES)
            ax.set_yticks(range(len(HZ_VALUES))); ax.set_yticklabels(HZ_VALUES)
            if r == 0:
                ax.set(title=ARM_LABEL[arm])
            else:
                ax.set(xlabel="$h_x$")
        axes[r][0].set(ylabel=f"{row_lab}\n$h_z$")
        if norm_r is not None:
            cb = fig.colorbar(im, ax=list(axes[r]), shrink=0.85, pad=0.02)
            cb.set_label("achieved infidelity $1-F$" if r == 0 else "exact ceiling $1-F_s$")
    fig.suptitle("Achieved trained infidelity (top) vs gate-0 sign-fidelity ceiling "
                 "(bottom), per arm", y=1.02)
    # plt.savefig(FIGS / "fermionic_plane_L2_invivo_vs_ceiling.png", dpi=300, bbox_inches="tight")
    plt.show()


plot_invivo_vs_ceiling()


## 6. Reading

**Data caveat.** The panels above currently run on the $h_z=0$ stand-in
ladder (`results/fermionic_hx_ladder/`), not the real 16-point plane
campaign — only the $h_z=0$ row has any achieved-training data (`asymm`,
`anaC_k0`; `pt2sf`/`pt2sfc` don't exist yet at any point), and only at
$h_x\in\{0.2,0.5,1.0\}$ (no $h_x=0$ run in the stand-in). Every other cell
is a genuine missing-data NaN, which is the behavior this notebook is meant
to validate before the real campaign lands. Once `results/fermionic_plane_L2/`
is populated, re-running this notebook unchanged fills in the full grid — no
edits needed beyond re-executing.

**What the stand-in slice already shows (consistent with
`fermionic_hx_ladder.ipynb`).** At $h_z=0$, `anaC_k0`'s frozen analytic head
wins at small $h_x$ (flux sectors are still nearly conserved) while `asymm`
overtakes at large $h_x$ as the field polarizes the state and the sign
structure becomes close to trivial for an unconstrained trunk. The §5 ceiling
row (computed on the *real* 4x4 grid via gate-0, independent of which NQS
data happen to be loaded above) is what determines where each arm's
achieved infidelity *can* go: the frozen `anaC` head's ceiling degrades
monotonically away from $h_x=h_z=0$ as flux sectors mix, while `plus`
(sign-blind guess, matching `asymm`'s floor with no head at all) sits far
higher across the whole plane except exactly at $h_x=0$ off-axis points where
the ground state is real and non-negative by symmetry.

**Gate-0 ceilings, $1-F_s$, on the real 4x4 grid**
(`results/fermionic_gate0/2x2x2_OBC_plane_gate0.json`, `anaC` / `pt2` /
`plus`; `exact` marks machine precision $\sim10^{-15}$):

- **`pt2` (perturbative, exact denominators) is essentially exact almost
  everywhere** on the plane -- it only leaves the $10^{-15}$ floor near
  $h_x=1$ (reaching $1-F_s\approx0.05$–$0.07$ at $h_x=1$) and picks up a tiny
  $10^{-5}$–$10^{-4}$ residual at $(h_x,h_z)=(0.2,1.0)$ and $(0.5,1.0)$. It is
  the best head at every single grid point.
- **`anaC` (frozen analytic, support-only) is exact along the whole
  $h_x=0$ column** for every $h_z$ (the star term alone doesn't move it off
  its designed support) but **degrades monotonically with $h_x$**,
  reaching $1-F_s\approx0.19$–$0.22$ at $h_x=1$ -- one to four orders of
  magnitude worse than `pt2` at the same point, and roughly flat in $h_z$ at
  fixed $h_x$.
- **`plus` (sign-blind, all-$+1$) is worst at $h_x=0$** ($1-F_s\approx
  0.23$–$0.38$ even at $h=0$, since the star term $A_v$ already carries a
  nontrivial sign) but **collapses to exact at $h_x=1$** for every $h_z$ --
  the fully $\sigma^x$-polarized ground state is real and sign-trivial, so a
  no-head trunk (`asymm`) faces its easiest sign problem exactly where
  `anaC` faces its hardest one.